# Football Tracker — Training & Evaluation Notebook

**Colab-ready. SoccerNet primary dataset (pip, no manual auth).**

1. Google Drive mount + environment setup
2. Download SoccerNet Tracking (pip install + download)
3. Convert MOT -> YOLO format
4. Fine-tune YOLO26s player detector (checkpoints -> Drive)
5. Train YOLO26n ball detector @ 1280px
6. Team classifier comparison (HSV vs ReID)
7. End-to-end pipeline evaluation
8. Export models to TensorRT / ONNX
9. Smoke test

## 0. Colab Setup — Google Drive + Dependencies

In [ ]:
# [Cell 0] — Mount Google Drive & install dependencies

import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/football-tracker")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_DIR = Path("drive_checkpoints")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoints: {DRIVE_DIR.resolve()}")

# !pip install -q ultralytics supervision opencv-python numpy scikit-learn pyyaml

import cv2, json, yaml, numpy as np
from ultralytics import YOLO
import supervision as sv

DATA_DIR = Path("data")
MODELS_DIR = DRIVE_DIR / "models"
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
print(f"Data:   {DATA_DIR.resolve()}")
print(f"Models: {MODELS_DIR.resolve()}")

## 1. Download SoccerNet Tracking Dataset

12 complete games, 200 annotated clips (30s each), 720p/25fps.
Player bboxes + track IDs in MOT format.

**Install the pip package and download — no manual authentication needed.**

In [ ]:
# [Cell 1] — Download SoccerNet Tracking (pip, no manual auth)

!pip install -q SoccerNet
from SoccerNet.Downloader import SoccerNetDownloader

downloader = SoccerNetDownloader(LocalDirectory=str(DATA_DIR / "SoccerNet"))

print("Downloading SoccerNet Tracking (train + test + challenge)...")
downloader.downloadDataTask(task="tracking", split=["train", "test", "challenge"])
print("Done.")

# Optional: GSR 2023 (richer annotations)
if False:  # Set True to download GSR too.
    downloader.downloadDataTask(task="tracking-2023", split=["train", "test", "challenge"])
    print("GSR 2023 downloaded.")

## 2. Convert Dataset to YOLO Format

SoccerNet uses MOT format (10-column CSV). Convert to YOLO txt format.
```
data/SoccerNet/tracking/train/
  clip_001.mp4    # video
  clip_001.csv    # ground truth
```

In [ ]:
# [Cell 2] — Convert MOT CSV -> YOLO txt format

def convert_mot_to_yolo(gt_path, video_path, output_dir, class_id=0, img_ext=".jpg"):
    """Convert MOT CSV + video to YOLO format (images/ + labels/).
    MOT CSV: frame, track_id, x, y, w, h, conf, -1, -1, -1
    YOLO txt: class_id cx cy cw ch (normalised 0..1)"""
    (output_dir / "images").mkdir(parents=True, exist_ok=True)
    (output_dir / "labels").mkdir(parents=True, exist_ok=True)
    if not video_path.exists():
        return 0
    gt_by_frame = {}
    for line in open(gt_path):
        p = line.strip().split(",")
        gt_by_frame.setdefault(int(p[0]), []).append(
            (float(p[2]), float(p[3]), float(p[4]), float(p[5])))
    cap = cv2.VideoCapture(str(video_path))
    w, h = int(cap.get(3)), int(cap.get(4))
    count = fid = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        fid += 1
        if fid not in gt_by_frame: continue
        img = f"{gt_path.stem}_{fid:06d}{img_ext}"
        cv2.imwrite(str(output_dir / "images" / img), frame)
        lines = []
        for (x, y, bw, bh) in gt_by_frame[fid]:
            cx = (x + bw/2) / w; cy = (y + bh/2) / h
            cw = bw / w; ch = bh / h
            lines.append(f"{class_id} {cx:.6f} {cy:.6f} {cw:.6f} {ch:.6f}")
        with open(output_dir / "labels" / f"{gt_path.stem}_{fid:06d}.txt", "w") as lf:
            lf.write("\n".join(lines))
        count += 1
    cap.release()
    return count


DO_CONVERT = False  # <- Set True after downloading.

if DO_CONVERT:
    tracking_train = DATA_DIR / "SoccerNet" / "tracking" / "train"
    output = DATA_DIR / "football_yolo"
    total = 0
    for csv_file in sorted(tracking_train.rglob("*.csv")):
        video_file = csv_file.with_suffix(".mp4")
        if not video_file.exists():
            video_file = csv_file.with_suffix(".mkv")
        n = convert_mot_to_yolo(csv_file, video_file, output)
        if n: print(f"  {csv_file.name}: {n} frames")
        total += n
    print(f"Total: {total} annotated frames")
    yaml.dump({
        "path": str(output.resolve()),
        "train": "images", "val": "images",
        "nc": 1, "names": ["player"],
    }, open(output / "football.yaml", "w"))
    print(f"Created {output / 'football.yaml'}")
else:
    print("Skipped. Set DO_CONVERT=True after downloading.")

## 3. Player Detector Fine-Tuning

Fine-tune YOLO26s. Checkpoints saved to Drive every 5 epochs.

In [ ]:
# [Cell 3] — Fine-tune YOLO26s (checkpoints -> Drive)

from ultralytics import settings
settings.update({"runs_dir": str(DRIVE_DIR / "runs")})

player_model = YOLO("yolo26s.pt")
DO_TRAIN = False  # <- Set True.

if DO_TRAIN:
    results = player_model.train(
        data=str(DATA_DIR / "football_yolo" / "football.yaml"),
        epochs=50, imgsz=640, batch=16,
        name="football_player", pretrained=True, device=0,
        save_period=5,
        hsv_h=0.015, hsv_s=0.3, hsv_v=0.2,
        degrees=5, translate=0.1, scale=0.5,
        fliplr=0.5, mosaic=1.0, mixup=0.1,
    )
    print(f"Best model: {results.save_dir}")
else:
    print(f"Skipped. Set DO_TRAIN=True. Checkpoints -> {DRIVE_DIR / 'runs'}")

In [ ]:
# [Cell 4] — Resume from last Drive checkpoint

DO_RESUME = False
if DO_RESUME:
    runs = sorted((DRIVE_DIR / "runs" / "detect").glob("football_player*"), reverse=True)
    if runs and (runs[0] / "weights" / "last.pt").exists():
        print(f"Resuming from {runs[0]}")
        YOLO(str(runs[0] / "weights" / "last.pt")).train(resume=True)
    else:
        print("No checkpoint found.")
else:
    print("Skipped.")

In [ ]:
# [Cell 5] — Copy best.pt to Drive models/

DO_SAVE = False
if DO_SAVE:
    import shutil
    runs = sorted((DRIVE_DIR / "runs" / "detect").glob("football_player*"), reverse=True)
    if runs and (runs[0] / "weights" / "best.pt").exists():
        shutil.copy(runs[0] / "weights" / "best.pt", MODELS_DIR / "player_detector.pt")
        print(f"Copied to {MODELS_DIR / 'player_detector.pt'}")
else:
    print("Skipped.")

## 4. Ball Detector Training

Train YOLO26n @ 1280px for small-ball detection.
Needs SoccerNet GSR (ball annotations) or manual labeling.

## 4a. Prepare Ball Training Data

SoccerNet GSR includes ball annotations in Labels-GameState.json
(pitch coordinates + track IDs). We extract ball bboxes from the tracking
CSV and convert to YOLO format at 1280px for small-object training.

**Prerequisite:** GSR 2023 must be downloaded (set  in Cell 1).

In [ ]:
# [Cell 6a] — Extract ball annotations from SoccerNet GSR

import json
from collections import defaultdict

def extract_ball_bboxes(gsr_dir, tracking_dir, output_dir):
    """Extract ball bounding boxes from GSR + tracking data.
    
    Reads Labels-GameState.json to find which track_id is the ball,
    then extracts the corresponding bboxes from the tracking CSV.
    """
    (output_dir / "images").mkdir(parents=True, exist_ok=True)
    (output_dir / "labels").mkdir(parents=True, exist_ok=True)
    
    total = 0
    for gsr_json in sorted(Path(gsr_dir).rglob("Labels-GameState.json")):
        with open(gsr_json) as f:
            gsr = json.load(f)
        
        # Find the matching tracking CSV
        clip_name = gsr_json.parent.name
        csv_path = Path(tracking_dir) / f"{clip_name}.csv"
        video_path = Path(tracking_dir) / f"{clip_name}.mp4"
        if not video_path.exists():
            video_path = Path(tracking_dir) / f"{clip_name}.mkv"
        if not csv_path.exists() or not video_path.exists():
            continue
        
        # Find ball track IDs from GSR
        ball_track_ids = set()
        for ann in gsr.get("annotations", []):
            if ann.get("role") == "ball":
                ball_track_ids.add(ann["track_id"])
        
        if not ball_track_ids:
            continue
        
        # Parse tracking CSV: frame -> [(track_id, x, y, w, h)]
        tracking = defaultdict(list)
        for line in open(csv_path):
            p = line.strip().split(",")
            tid = int(p[1])
            if tid in ball_track_ids:
                tracking[int(p[0])].append((float(p[2]), float(p[3]), float(p[4]), float(p[5])))
        
        # Extract frames with ball visible
        cap = cv2.VideoCapture(str(video_path))
        w, h = int(cap.get(3)), int(cap.get(4))
        fid = 0
        while True:
            ok, frame = cap.read()
            if not ok: break
            fid += 1
            if fid not in tracking: continue
            
            for (bx, by, bw, bh) in tracking[fid]:
                if bw < 3 or bh < 3: continue  # skip tiny detections
                img_name = f"{clip_name}_{fid:06d}.jpg"
                cv2.imwrite(str(output_dir / "images" / img_name), frame)
                cx = (bx + bw/2) / w
                cy = (by + bh/2) / h
                cw = bw / w
                ch = bh / h
                with open(output_dir / "labels" / f"{clip_name}_{fid:06d}.txt", "w") as lf:
                    lf.write(f"0 {cx:.6f} {cy:.6f} {cw:.6f} {ch:.6f}")
                total += 1
        cap.release()
    
    return total


DO_PREP_BALL = False  # <- Set True after downloading GSR.

if DO_PREP_BALL:
    gsr_dir = DATA_DIR / "SoccerNet" / "tracking-2023" / "train"
    tracking_dir = DATA_DIR / "SoccerNet" / "tracking" / "train"
    output = DATA_DIR / "ball_yolo"
    n = extract_ball_bboxes(gsr_dir, tracking_dir, output)
    print(f"Extracted {n} ball frames")
    
    yaml.dump({
        "path": str(output.resolve()),
        "train": "images", "val": "images",
        "nc": 1, "names": ["ball"],
    }, open(output / "ball.yaml", "w"))
    print(f"Created {output / 'ball.yaml'}")
else:
    print("Skipped. Set DO_PREP_BALL=True after downloading GSR.")

In [ ]:
# [Cell 6] — Train YOLO26n ball detector @ 1280px

ball_model = YOLO("yolo26n.pt")
DO_TRAIN_BALL = False
if DO_TRAIN_BALL:
    results = ball_model.train(
        data=str(DATA_DIR / "ball_yolo" / "ball.yaml"),
        epochs=100, imgsz=1280, batch=8,
        name="football_ball", pretrained=True, device=0, save_period=10,
        degrees=2, translate=0.05, scale=0.3, fliplr=0.0, mosaic=0.5, erasing=0.2,
    )
    print(f"Best model: {results.save_dir}")
else:
    print("Skipped. Needs ball_yolo dataset.")

## 5. Team Classifier Comparison

HSV clustering (fast, no training) vs ReID embedding (robust).

In [ ]:
# [Cell 7] — Extract player torso crops & HSV team classifier

sys.path.insert(0, str(Path.cwd()))
from src.service.team_classifier import TeamClassifier
from collections import Counter

def extract_player_crops(video_path, n_crops=200):
    detector = YOLO("yolo26s.pt")
    cap = cv2.VideoCapture(video_path)
    crops = []
    while len(crops) < n_crops:
        ok, frame = cap.read()
        if not ok: break
        for r in detector(frame, classes=[0], verbose=False):
            if r.boxes is None: continue
            for box in r.boxes.xyxy:
                x1, y1, x2, y2 = map(int, box)
                crop = frame[y1+(y2-y1)//2:y2, x1:x2]
                if crop.size > 0: crops.append(crop)
                if len(crops) >= n_crops: break
    cap.release()
    return crops

VIDEO_PATH = "data/sample.mp4"
if Path(VIDEO_PATH).exists():
    crops = extract_player_crops(VIDEO_PATH)
    print(f"Extracted {len(crops)} crops")
    clf = TeamClassifier(n_clusters=3)
    clf.fit(crops)
    labels = [clf.predict(c) for c in crops]
    print(f"Clusters: {Counter(labels)}")
else:
    print(f"{VIDEO_PATH} not found.")

In [ ]:
# [Cell 8] — ReID comparison (optional)

DO_REID = False
if DO_REID:
    print("Steps: 1) yolo export model=yolo26n-cls.pt format=onnx")
    print("       2) Extract embeddings from each crop")
    print("       3) K-means on embeddings, compare ARI with HSV")
else:
    print("Skipped.")

## 6. End-to-End Pipeline Evaluation

Run pipeline on SoccerNet test clips, compare with ground truth.

In [ ]:
# [Cell 9] — Evaluate on SoccerNet test clips

from src.controller.pipeline import Pipeline

def evaluate(video_path, gt_csv_path, config):
    pipeline = Pipeline(config)
    frames = pipeline.run(video_path)
    gt_data = {}
    if Path(gt_csv_path).exists():
        for line in open(gt_csv_path):
            p = line.strip().split(",")
            gt_data.setdefault(int(p[0]), []).append({
                "track_id": int(p[1]),
                "bbox": (float(p[2]), float(p[3]),
                         float(p[2])+float(p[4]), float(p[3])+float(p[5])),
            })
    print(f"Frames: {len(frames)}, GT: {sum(len(v) for v in gt_data.values())}")
    return {"frames": len(frames), "gt_count": sum(len(v) for v in gt_data.values())}

DO_EVAL = False  # <- Set True.
if DO_EVAL:
    cfg = yaml.safe_load(open("config/default.yaml"))
    cfg["pipeline"]["display"] = False
    cfg["team_classifier"]["enabled"] = False
    cfg["calibration"]["enabled"] = False
    test_dir = DATA_DIR / "SoccerNet" / "tracking" / "test"
    clips = sorted(test_dir.glob("*.mp4"))
    if clips:
        clip = clips[0]
        metrics = evaluate(str(clip), str(clip.with_suffix(".csv")), cfg)
        print(json.dumps(metrics, indent=2))
    else:
        print("No test clips. Run Cell 1 first.")
else:
    print("Skipped.")

## 7. Export Models for Production

TensorRT FP16: ~5x speedup on GPU. ONNX: portable.

In [ ]:
# [Cell 10] — Export to TensorRT / ONNX

DO_EXPORT = False
if DO_EXPORT:
    import shutil
    m = YOLO(str(MODELS_DIR / "player_detector.pt"))
    m.export(format="engine", quantize=16, device=0)
    m = YOLO(str(MODELS_DIR / "ball_detector.pt"))
    m.export(format="onnx")
    for f in Path.cwd().glob("*.engine"): shutil.copy(f, MODELS_DIR / f.name)
    for f in Path.cwd().glob("*.onnx"): shutil.copy(f, MODELS_DIR / f.name)
    print(f"Exported to {MODELS_DIR}")
else:
    print("Skipped.")

## Quick Smoke Test

In [ ]:
# [Cell 11] — Smoke test

SMOKE_TEST = False
if SMOKE_TEST:
    cfg = yaml.safe_load(open("config/default.yaml"))
    cfg["model"]["device"] = "cpu"
    cfg["pipeline"]["display"] = False
    cfg["calibration"]["enabled"] = False
    frames = Pipeline(cfg).run(cfg["video"]["source"])
    print(f"Pipeline OK - {len(frames)} frames.")
else:
    print("Skipped.")